# Impute solvents from a processed NPZ

This notebook loads a processed structure `.npz`, optionally strips existing solvents, imputes additional solvents with `Structure.impute_solvents(...)`, and writes the result as an mmCIF file.

Edit the config cell below, then run the notebook top to bottom.

In [12]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np
from boltzgen.data.data import Structure
from boltzgen.data.write.mmcif import to_mmcif


BOLTZGEN_SRC = Path("/data/rbg/users/gloriama/dev/foldeverything/src")
NOTEBOOK_ROOT = Path("/data/rbg/users/gloriama/dev/water_conservation")


from gloria_hbond_helpers import (
    gloria_get_solvent_hbond_counts,
    gloria_get_solvent_hbond_counts_and_mask,
    gloria_get_solvent_hbond_mask,
)
from unused_solvent_filters import (
    gloria_remove_low_b_factor_solvents,
    gloria_remove_weak_solvents,
)
from impute_solvents_with_num_hbonds import impute_solvents_with_num_hbonds
from basic_helpers import resolve_npz_path
from filter_solvent_clashes import filter_solvent_clashes
from impute_solvents_from_triples import *



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
# Config 
NPZ_ROOT = Path("/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures")
NPZ_PATH = None  # Set to an explicit file path to override PDB_ID + NPZ_ROOT.
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs"
ONE_SOLVENT_PER_CHAIN = True


# Unused for new imputation functions
MIN_SOLVENTS = 20
STRIP_SOLVENTS_BEFOREHAND = True
ALLOW_OVERLAP_WITH_REAL_SOLVENTS = False
MAX_SAMPLE_ATTEMPTS = 1000


# Change 
PDB_ID = "1ubq"
COLLISION_MIN_DIST = 2.5 # ? 
# TRIPLE_HBOND_PAIR_MAX_DIST = 8.0
npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
gt_structure = Structure.load(npz_path)
gt_structure = gt_structure.to_one_solvent_per_chain(gt_structure)
gt_solvent_count = gt_structure.count_solvents()
stripped_gt = gt_structure.remove_solvents()
gt_solvent_count

def stripped_gt_structure(PDB_ID):
    npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
    gt_structure = Structure.load(npz_path)
    gt_structure = gt_structure.to_one_solvent_per_chain(gt_structure)
    stripped_gt = gt_structure.remove_solvents()
    return stripped_gt


# 3 hbond exhaustive imputation

In [ ]:


atom_triples = find_atom_triples_for_three_hbonds(
    gt_structure,
    max_pair_dist=TRIPLE_HBOND_PAIR_MAX_DIST,
)
imputed = impute_solvents_from_atom_triples(
    gt_structure,
    one_solvent_per_chain=True,
    max_pair_dist=TRIPLE_HBOND_PAIR_MAX_DIST,
)
no_collisions = filter_solvent_clashes(
    imputed,
    min_dist=COLLISION_MIN_DIST,
)

output_path = OUTPUT_DIR / (
    f"{PDB_ID}_all_triples_stripped_filtered_maxdist_{TRIPLE_HBOND_PAIR_MAX_DIST}.cif"
)
output_path.write_text(to_mmcif(no_collisions))

print(f"Input NPZ: {npz_path}")
print(f"Triple max pair distance: {TRIPLE_HBOND_PAIR_MAX_DIST}")
print(f"Triple count: {len(atom_triples)}")
print(f"Placed waters before filtering: {imputed.count_solvents()}")
print(f"Placed waters after filtering: {no_collisions.count_solvents()}")
print(f"Wrote CIF to: {triple_output_path}")

In [21]:
# from impute_solvents_from_triples import get_hbond_candidate_atom_data

hbond_candidate_indices, hbond_candidate_coords = get_hbond_candidate_atom_data(gt_structure)
print(len(hbond_candidate_indices))
print(len(hbond_candidate_coords))
print(hbond_candidate_indices)


281
281
[  0   3   6   8  11  15  16  17  20  25  28  36  39  43  46  51  52  55
  57  59  62  67  70  72  74  77  78  81  86  87  90  92  94  97 102 105
 107 109 112 117 120 124 125 126 129 133 136 140 141 142 145 149 152 154
 155 158 161 162 163 166 168 170 173 178 181 185 186 187 190 193 194 195
 198 202 205 210 211 214 216 219 224 225 228 233 236 240 241 242 245 248
 249 250 253 258 259 262 266 267 268 271 272 275 280 283 287 290 294 297
 300 301 302 305 309 310 311 314 318 319 320 323 327 329 330 331 334 339
 342 347 350 358 361 363 366 367 370 375 376 379 383 384 385 388 393 396
 400 401 402 405 408 409 410 413 414 417 421 423 424 425 428 430 432 435
 440 443 445 446 449 452 453 454 457 465 466 469 472 473 474 477 482 485
 489 490 491 494 499 500 503 507 508 509 512 514 515 518 520 522 525 530
 533 536 539 540 543 548 551 555 558 563 566 570 572 573 574 577 582 585
 589 591 592 593 596 597 600 601 602 603 604 605 606 607 608 609 610 611
 612 613 614 615 616 617 618 619 620 621 62

In [27]:
kd_triple_list = kdtree_find_atom_triples_for_three_hbonds(
    gt_structure,
    max_pair_dist=7.0,
)
print(len(kd_triple_list))
print(kd_triple_list[:10])

Number of neighbor lists: 281
Neighbor lists: [[0, 1, 2, 3, 4, 7, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 231, 245, 257, 262, 265, 274], [0, 1, 2, 3, 4, 7, 8, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 185, 231, 245, 262], [0, 1, 2, 3, 4, 7, 43, 44, 45, 46, 47, 49, 177, 178, 179, 182, 183, 185, 189, 191, 231, 244, 245, 265, 270], [0, 1, 2, 3, 4, 5, 6, 7, 8, 37, 38, 39, 40, 41, 43, 44, 179, 182, 183, 185, 188, 231, 245, 262], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 37, 38, 39, 43, 44, 179, 182, 183, 185, 186, 187, 188, 189, 190, 191, 245], [3, 4, 5, 6, 7, 8, 34, 35, 36, 37, 38, 41, 188, 245], [3, 4, 5, 6, 7, 8, 36, 38, 185, 187, 188, 245, 263], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 40, 43, 44, 179, 185, 186, 188, 189, 190, 245], [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 32, 33, 34, 35, 36, 37, 38, 39, 43, 185, 186, 189, 190, 192], [4, 7, 8, 9, 10, 11, 12, 13, 32, 33, 34, 35, 37, 38, 185, 186, 189, 190, 192, 193, 194, 195, 196]]
9752
[[  0   3   6]
 [  0   3   8]


In [18]:
old_triple_list = find_atom_triples_for_three_hbonds(
    gt_structure,
    max_pair_dist=7.0,
)
print(len(old_triple_list))
print(old_triple_list[:10])



784
[[  0   6   8]
 [  0   6 643]
 [  0   8 126]
 [  0 126 133]
 [  0 133 140]
 [  0 133 141]
 [  0 140 643]
 [  0 141 652]
 [  3   6  11]
 [  3  11 112]]


In [26]:
TRIPLE_HBOND_PAIR_MAX_DIST = 7.0
imputed = impute_solvents_from_atom_triples(
    gt_structure,
    one_solvent_per_chain=True,
    max_pair_dist=7.0,
)
no_collisions = filter_solvent_clashes(
    imputed,
    min_dist=COLLISION_MIN_DIST,
)

output_path = OUTPUT_DIR / (
    f"{PDB_ID}_all_KD_triples_stripped_filtered_maxdist_{TRIPLE_HBOND_PAIR_MAX_DIST}.cif"
)
output_path.write_text(to_mmcif(no_collisions))

print(f"Input NPZ: {npz_path}")
print(f"Triple max pair distance: {TRIPLE_HBOND_PAIR_MAX_DIST}")
print(f"Placed waters before filtering: {imputed.count_solvents()}")
print(f"Placed waters after filtering: {no_collisions.count_solvents()}")
print(f"Wrote CIF to: {output_path}")

Number of neighbor lists: 281
Neighbor lists: [[0, 1, 2, 3, 4, 7, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 231, 245, 257, 262, 265, 274], [0, 1, 2, 3, 4, 7, 8, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 185, 231, 245, 262], [0, 1, 2, 3, 4, 7, 43, 44, 45, 46, 47, 49, 177, 178, 179, 182, 183, 185, 189, 191, 231, 244, 245, 265, 270], [0, 1, 2, 3, 4, 5, 6, 7, 8, 37, 38, 39, 40, 41, 43, 44, 179, 182, 183, 185, 188, 231, 245, 262], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 37, 38, 39, 43, 44, 179, 182, 183, 185, 186, 187, 188, 189, 190, 191, 245], [3, 4, 5, 6, 7, 8, 34, 35, 36, 37, 38, 41, 188, 245], [3, 4, 5, 6, 7, 8, 36, 38, 185, 187, 188, 245, 263], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 40, 43, 44, 179, 185, 186, 188, 189, 190, 245], [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 32, 33, 34, 35, 36, 37, 38, 39, 43, 185, 186, 189, 190, 192], [4, 7, 8, 9, 10, 11, 12, 13, 32, 33, 34, 35, 37, 38, 185, 186, 189, 190, 192, 193, 194, 195, 196]]
Input NPZ: /data/rbg/shared/dataset

In [40]:
TRIPLE_HBOND_PAIR_MAX_DIST = 7.0
imputed = impute_solvents_from_atom_triples(
    gt_structure,
    one_solvent_per_chain=True,
    max_pair_dist=7.0,
)
no_collisions = filter_solvent_clashes(
    imputed,
    min_dist=COLLISION_MIN_DIST,
)

output_path = OUTPUT_DIR / (
    f"{PDB_ID}_all_KD_triples_stripped_filtered_maxdist_{TRIPLE_HBOND_PAIR_MAX_DIST}.cif"
)
output_path.write_text(to_mmcif(no_collisions))

print(f"Input NPZ: {npz_path}")
print(f"Triple max pair distance: {TRIPLE_HBOND_PAIR_MAX_DIST}")
print(f"Placed waters before filtering: {imputed.count_solvents()}")
print(f"Placed waters after filtering: {no_collisions.count_solvents()}")
print(f"Wrote CIF to: {output_path}")

Number of neighbor lists: 281
Neighbor lists: [[0, 1, 2, 3, 4, 7, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 231, 245, 257, 262, 265, 274], [0, 1, 2, 3, 4, 7, 8, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 48, 185, 231, 245, 262], [0, 1, 2, 3, 4, 7, 43, 44, 45, 46, 47, 49, 177, 178, 179, 182, 183, 185, 189, 191, 231, 244, 245, 265, 270], [0, 1, 2, 3, 4, 5, 6, 7, 8, 37, 38, 39, 40, 41, 43, 44, 179, 182, 183, 185, 188, 231, 245, 262], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 37, 38, 39, 43, 44, 179, 182, 183, 185, 186, 187, 188, 189, 190, 191, 245], [3, 4, 5, 6, 7, 8, 34, 35, 36, 37, 38, 41, 188, 245], [3, 4, 5, 6, 7, 8, 36, 38, 185, 187, 188, 245, 263], [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 33, 34, 35, 36, 37, 38, 39, 40, 43, 44, 179, 185, 186, 188, 189, 190, 245], [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 32, 33, 34, 35, 36, 37, 38, 39, 43, 185, 186, 189, 190, 192], [4, 7, 8, 9, 10, 11, 12, 13, 32, 33, 34, 35, 37, 38, 185, 186, 189, 190, 192, 193, 194, 195, 196]]
Input NPZ: /data/rbg/shared/dataset

In [37]:
TRIPLE_HBOND_PAIR_MAX_DIST = 7.0
gt_structure = stripped_gt_structure("7zzh")


imputed = impute_solvents_from_atom_triples(
    gt_structure,
    one_solvent_per_chain=True,
    max_pair_dist=7.0,
)
no_collisions = filter_solvent_clashes(
    imputed,
    min_dist=COLLISION_MIN_DIST,
)

output_path = OUTPUT_DIR / (
    f"{"7zzh"}_all_KD_triples_stripped_filtered_maxdist_{TRIPLE_HBOND_PAIR_MAX_DIST}.cif"
)
output_path.write_text(to_mmcif(no_collisions))

print(f"Input NPZ: {npz_path}")
print(f"Triple max pair distance: {TRIPLE_HBOND_PAIR_MAX_DIST}")
print(f"Placed waters before filtering: {imputed.count_solvents()}")
print(f"Placed waters after filtering: {no_collisions.count_solvents()}")
print(f"Wrote CIF to: {output_path}")

Number of neighbor lists: 3072
Neighbor lists: [[0, 1, 2, 3, 4, 6, 78, 79, 80, 81, 82, 83, 84, 85, 86, 110, 255, 256, 257], [0, 1, 2, 3, 4, 6, 7, 79, 80, 82, 83, 84, 85, 86, 87, 89, 92, 108, 110, 112], [0, 1, 2, 3, 4, 84, 108, 109, 110, 111, 112, 255, 257, 261, 267], [0, 1, 2, 3, 4, 5, 6, 7, 79, 82, 83, 84, 85, 86, 87, 89, 92, 93, 103, 106, 107, 108, 110, 111, 112], [0, 1, 2, 3, 4, 6, 7, 9, 84, 85, 86, 89, 92, 93, 103, 106, 107, 108, 109, 110, 111, 112, 113, 114], [3, 5, 86, 87, 88, 90, 92, 94, 95, 102, 103, 104], [0, 1, 3, 4, 6, 7, 8, 9, 10, 83, 84, 85, 86, 87, 89, 90, 92, 93, 94, 95, 96, 103, 106, 107, 108, 109, 112, 113, 114], [1, 3, 4, 6, 7, 8, 9, 10, 11, 12, 85, 86, 89, 90, 91, 92, 93, 94, 96, 100, 106, 108, 112, 113, 114], [6, 7, 8, 9, 12, 48, 59, 61, 62, 63, 64, 65, 67, 68, 69, 83, 85, 86, 89, 90, 92], [4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 86, 89, 90, 91, 92, 93, 96, 106, 107, 108, 112, 113, 114, 115, 116, 117, 157]]
Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvent

In [32]:
weak_removed_from_imputed = gloria_remove_weak_solvents(no_collisions)
weak_removed_from_imputed.count_solvents()


94

# Save cifs (gt without imputation, with filtering.)

In [ ]:
default_pdb_id = "1ubq"
default_npz_path = resolve_npz_path(default_pdb_id, NPZ_ROOT)
default_output_dir = Path(OUTPUT_DIR)
default_output_dir.mkdir(parents=True, exist_ok=True)

default_structure = Structure.load(default_npz_path)
default_output_path = default_output_dir / f"{default_pdb_id}_default.cif"
default_output_path.write_text(to_mmcif(default_structure))

print(f"Input NPZ: {default_npz_path}")
print(f"Wrote CIF to: {default_output_path}")

In [ ]:
weak_pdb_id = "1ubq"
weak_npz_path = resolve_npz_path(weak_pdb_id, NPZ_ROOT)
weak_output_dir = Path(OUTPUT_DIR)
weak_output_dir.mkdir(parents=True, exist_ok=True)
weak_structure = Structure.load(weak_npz_path)
weak_structure = weak_structure.to_one_solvent_per_chain(weak_structure)
weak_filtered_structure = gloria_remove_weak_solvents(weak_structure)
weak_output_path = weak_output_dir / f"{weak_pdb_id}_weak_filtered.cif"
weak_output_path.write_text(to_mmcif(weak_filtered_structure))
print(f"Input NPZ: {weak_npz_path}")
print(f"Original solvent count: {weak_structure.count_solvents()}")
print(f"Filtered solvent count: {weak_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak_output_path}")


In [ ]:
weak3_pdb_id = "1ubq"
weak3_npz_path = resolve_npz_path(weak3_pdb_id, NPZ_ROOT)
weak3_output_dir = Path(OUTPUT_DIR)
weak3_output_dir.mkdir(parents=True, exist_ok=True)
weak3_structure = Structure.load(weak3_npz_path)
weak3_structure = weak3_structure.to_one_solvent_per_chain(weak3_structure)
weak3_filtered_structure = gloria_remove_weak_solvents(
    weak3_structure,
    min_hbonds=3,
)
weak3_output_path = weak3_output_dir / f"{weak3_pdb_id}_weak_filtered_min3.cif"
weak3_output_path.write_text(to_mmcif(weak3_filtered_structure))
print(f"Input NPZ: {weak3_npz_path}")
print(f"Original solvent count: {weak3_structure.count_solvents()}")
print(f"Filtered solvent count (min_hbonds=3): {weak3_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak3_output_path}")


## Old imputation function (sample 3, 2, 1, etc.)
- Bug: was only attempting starting at 2, 1, etc. 
- For some reason only scores 1hbond waters, even with max_sample_attempts = 1000. 

In [4]:
npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not npz_path.exists():
    raise FileNotFoundError(f"Could not find NPZ input: {npz_path}")

structure = Structure.load(npz_path)
loaded_solvents = structure.count_solvents()

if STRIP_SOLVENTS_BEFOREHAND:
    structure = structure.remove_solvents()

post_strip_solvents = structure.count_solvents()
structure = impute_solvents_with_num_hbonds(
    structure,
    one_solvent_per_chain=ONE_SOLVENT_PER_CHAIN,
    min_solvents=MIN_SOLVENTS,
    interaction_min_dist=INTERACTION_MIN_DIST,
    max_sample_attempts=MAX_SAMPLE_ATTEMPTS,
    allow_overlap_with_real_solvents=ALLOW_OVERLAP_WITH_REAL_SOLVENTS,
)
final_solvents = structure.count_solvents()

output_path = output_dir / f"{PDB_ID.lower()}_imputed_first_stripped.cif"
output_path.write_text(to_mmcif(structure))

print(f"Input NPZ: {npz_path}")
print(f"Loaded solvent count: {loaded_solvents}")
print(f"Post-strip solvent count: {post_strip_solvents}")
print(f"Final solvent count: {final_solvents}")
print(f"Wrote CIF to: {output_path}")


sample_attempts=10003 until a n_function(sample_attempts)=1 h-bond thing was found, for the impute_idx=0th water.
sample_attempts=10001 until a n_function(sample_attempts)=1 h-bond thing was found, for the impute_idx=1th water.


KeyboardInterrupt: 

In [ ]:
# Example helper calls
example_structure = Structure.load(npz_path)
example_structure = example_structure.to_one_solvent_per_chain(example_structure)

gloria_hbond_counts = gloria_get_solvent_hbond_counts(example_structure)
(
    gloria_hbond_counts_full,
    gloria_present_solvent_chain_mask,
) = gloria_get_solvent_hbond_counts_and_mask(example_structure)
gloria_keep_solvent_mask = gloria_get_solvent_hbond_mask(
    example_structure,
    min_hbonds=2,
)

gloria_weak_filtered_structure = gloria_remove_weak_solvents(
    example_structure,
    min_hbonds=2,
)
gloria_bfactor_filtered_structure = gloria_remove_low_b_factor_solvents(
    example_structure,
    n_keep=10,
)

print(f"Present solvent chains: {gloria_present_solvent_chain_mask.sum()}")
print(f"Per-chain hbond counts shape: {gloria_hbond_counts.shape}")
print(
    "Counts agree across helper variants:",
    (gloria_hbond_counts == gloria_hbond_counts_full).all(),
)
print(f"Chains meeting min_hbonds=2: {gloria_keep_solvent_mask.sum()}")
print(
    f"Solvents after gloria_remove_weak_solvents: {gloria_weak_filtered_structure.count_solvents()}"
)
print(
    f"Solvents after gloria_remove_low_b_factor_solvents(n_keep=10): {gloria_bfactor_filtered_structure.count_solvents()}"
)

Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Triple max pair distance: 8.0
Triple count: 550
Placed waters before filtering: 550
Placed waters after filtering: 9
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_all_triples_stripped_filtered_maxdist_8p0.cif


Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_default.cif


Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Original solvent count: 58
Filtered solvent count: 17
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_weak_filtered.cif


Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Original solvent count: 58
Filtered solvent count (min_hbonds=3): 3
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_weak_filtered_min3.cif
